# 08 · Priorização de iniciativas e seleção da solução — Vértice Retail

**O que este notebook decide:** onde a Vértice deve atuar primeiro, e qual capacidade deve
construir para sustentar essa captura.

**Princípio que governa o desenho:** a conclusão é *saída* do cálculo, nunca entrada. Não
existe neste notebook nenhum campo que diga de antemão em que quadrante uma iniciativa deve
cair ou qual solução deve vencer. Se o resultado contrariar a intuição da squad, o resultado
é reportado como está.

**De onde vêm os números:** exclusivamente de `outputs/numeros_canonicos.json` (fonte única,
gerada pelo `06`) e `outputs/impacto.json` (vereditos e naturezas, gerados pelo `07`).
Nenhum valor em R$ é digitado aqui — todos são buscados por chave, e a chave fica registrada
na saída para permitir auditoria linha a linha.

**O que este notebook deliberadamente não faz:** não calcula payback nem ROI. Payback exige
investimento, e investimento não existe em nenhuma base — só passa a existir depois que a
solução tem escopo definido. Estimar o denominador para produzir uma métrica de manchete
seria repetir o erro que este redesenho corrige. O retorno está medido; o investimento é
dimensionado no roadmap, na etapa seguinte.

In [1]:
import json, math
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 190)

OUT = Path('outputs')
CANON = json.loads((OUT / 'numeros_canonicos.json').read_text(encoding='utf-8'))
IMPACTO = json.loads((OUT / 'impacto.json').read_text(encoding='utf-8'))
MET = CANON['metricas']

USADAS = {}
def K(chave):
    """Busca um valor canônico e registra a chave usada (rastreabilidade)."""
    if chave not in MET:
        raise KeyError('chave canônica inexistente: ' + chave)
    USADAS[chave] = MET[chave]['valor']
    return MET[chave]['valor']

REGUA = K('margem_realizada_ano')
print('Fonte:', CANON['_meta']['gerado_por'], '|', len(MET), 'métricas canônicas')
print('Régua (denominador): margem de contribuição realizada anual = R$ {:,.0f}'.format(REGUA))
print('Janela:', CANON['_meta']['janela_vendas_dias'], 'dias | projeção base completa:', CANON['_meta']['projecao_base_completa'])

Fonte: 06_numeros_canonicos.ipynb | 98 métricas canônicas
Régua (denominador): margem de contribuição realizada anual = R$ 7,212,700
Janela: 390 dias | projeção base completa: False


## 0. A lente única: R$/ano de EBITDA

O diagnóstico produziu números de três naturezas contábeis diferentes — margem de
contribuição, receita e custo operacional. Ranqueá-los na mesma coluna sem conversão
superdimensiona quem está medido em receita e subdimensiona quem está medido em custo.

A lente única é **impacto em EBITDA por ano**, porque R$ 1 de margem recuperada e R$ 1 de
custo evitado valem exatamente o mesmo para o resultado. As regras de conversão:

| Natureza do número | Conversão para EBITDA | Por quê |
|---|---|---|
| Margem de contribuição recuperada | 1:1 | já está na linha de resultado |
| Custo operacional evitado | 1:1 | reduz despesa na mesma proporção |
| Receita protegida | × margem **observada daqueles SKUs** | receita não é resultado; a margem daquele mix é |

A terceira regra é o ponto onde o diagnóstico anterior errava: aplicava-se receita direto no
ranking. A correção não é multiplicar por uma margem média — isso seria um proxy. É somar a
margem que **aqueles mesmos SKUs** realizaram, que está na base (seção 12 do `06`).

### Sobre haircuts

Este notebook **não aplica fator de desconto sobre o R$**. Um número que a base mede é o que
a base mede; multiplicá-lo por 0,6 ou 0,4 para "ser conservador" reintroduz exatamente o tipo
de constante arbitrária que o redesenho existe para eliminar.

A incerteza sobre *quanto do valor é efetivamente capturável* é real, e ela entra no critério
de **risco** — que é declaradamente um juízo, numa escala de 0 a 10, e passa por teste de
sensibilidade. A regra é: não corromper o número de dinheiro; colocar a incerteza onde
incerteza pertence.

## 1. Das 14 hipóteses às iniciativas — por regra, não por curadoria

A lista de iniciativas é **derivada** do placar de hipóteses por uma regra escrita, aplicada
por código. Isso impede que uma frente confirmada desapareça da lista e que uma frente
refutada entre por simpatia.

| Veredito | Regra de inclusão |
|---|---|
| **Confirmada** | gera uma iniciativa por **endereço distinto de intervenção** — se a evidência aponta buracos que se resolvem por caminhos diferentes, são iniciativas diferentes |
| **Parcial** | gera iniciativa restrita ao recorte que a evidência sustenta |
| **Com ressalva** | gera iniciativa restrita ao que a evidência sustenta, nunca ao que ela sugere |
| **Não testável** | gera iniciativa de **instrumentação** (destravar a medição) — e, se houver achado colateral medido no mesmo segmento, uma iniciativa de captura para ele |
| **Refutada** | **não gera iniciativa.** Registra-se a decisão de não investir |

A regra sobre Confirmada é o que muda mais em relação à versão anterior. A H9 carrega
R$ 2,40 mi, mas esse número mede *tudo que não virou caixa* — devolução, cancelamento e
pagamento pendente. São três buracos com causas e donos diferentes: tratá-los como uma
iniciativa só produz uma linha grande e inacionável. A decomposição da seção 13 do `06`
permite endereçá-los separadamente, sem abrir hipótese nova.

In [2]:
REGRAS = {
    'Confirmada':   'uma iniciativa por endereço distinto de intervenção',
    'Parcial':      'iniciativa restrita ao recorte que a evidência sustenta',
    'Com ressalva': 'iniciativa restrita ao que a evidência sustenta',
    'Não testável': 'instrumentação para destravar + captura de achado colateral medido',
    'Refutada':     'nenhuma iniciativa; registra decisão de não investir',
}

HIP = {h['id']: h for h in IMPACTO['hipoteses']}
placar = IMPACTO['placar']
print('Placar do diagnóstico:', placar)
print('Total de hipóteses:', len(HIP))
for v, r in REGRAS.items():
    print('  {:14} ({}x)  ->  {}'.format(v, placar.get(v, 0), r))

Placar do diagnóstico: {'Refutada': 4, 'Confirmada': 5, 'Não testável': 3, 'Parcial': 1, 'Com ressalva': 1}
Total de hipóteses: 14
  Confirmada     (5x)  ->  uma iniciativa por endereço distinto de intervenção
  Parcial        (1x)  ->  iniciativa restrita ao recorte que a evidência sustenta
  Com ressalva   (1x)  ->  iniciativa restrita ao que a evidência sustenta
  Não testável   (3x)  ->  instrumentação para destravar + captura de achado colateral medido
  Refutada       (4x)  ->  nenhuma iniciativa; registra decisão de não investir


### 1.1 Catálogo derivado

Cada linha declara: a hipótese de origem, a **chave canônica** de onde o R$ sai (nunca o
valor digitado), a natureza contábil, a capacidade necessária para capturar, e os atributos
observáveis que alimentam esforço, velocidade e risco.

`chave_rs = None` significa **sem R$ mensurável** — a iniciativa entra na lista, mas com
impacto financeiro zero. É o tratamento correto para habilitadores: eles não são invisíveis,
mas também não recebem um valor inventado.

In [3]:
# mecanismo -> (pessoa_dias, dias_ate_primeiro_resultado)
# Arquétipos de execução. O esforço é declarado em pessoa-dias (unidade observável);
# nenhum valor em R$ é atribuído aqui - ver nota de abertura sobre payback.
MECANISMOS = {
    'definicao_metrica':    (12, 30),   # recalcular e publicar um indicador
    'regra_politica':       (15, 30),   # mudar uma regra comercial e aplicá-la no pedido
    'fila_operacional':     (20, 30),   # criar rotina de follow-up sobre uma fila existente
    'negociacao_comercial': (20, 60),   # renegociar contrato com terceiro
    'alerta_operacional':   (25, 45),   # monitorar posição e disparar ação
    'roteamento_existente': (10, 30),   # redirecionar volume para canal já em produção
    'regra_triagem':        (15, 45),   # classificar por regra determinística
    'informacao_produto':   (30, 60),   # corrigir conteúdo/ficha que gera expectativa errada
    'processo_fornecedor':  (40, 90),   # atuar em qualidade na origem, com terceiro
    'rotina_decisao':       (30, 45),   # instituir ritual de decisão sobre indicadores
    'instrumentacao_dado':  (35, 90),   # criar dado que hoje não existe
    'campanha_segmentada':  (25, 60),   # acionar base segmentada
}

# Cada iniciativa: id, título, hipótese, chave canônica do R$, natureza, capacidade,
# mecanismo, bases exigidas e atributos de risco.
CATALOGO = [
 dict(id='I01', titulo='Redefinir a métrica de margem realizada', hip='H9',
      chave_rs=None, natureza='habilitador', capacidade='regua_de_margem',
      mecanismo='definicao_metrica', bases=['vendas'],
      dep_terceiro=False, dado_novo=False, requer_adocao=True, muda_processo=True, reversivel=True,
      nota='Recálculo contábil. Torna a perda visível; não a recupera. Impacto financeiro direto = 0.'),

 dict(id='I02', titulo='Recuperação de pedidos com pagamento pendente', hip='H9',
      chave_rs='h9_aguardando_margem_ano', natureza='perda medida', capacidade='recuperacao_pos_venda',
      mecanismo='fila_operacional', bases=['vendas'],
      dep_terceiro=False, dado_novo=False, requer_adocao=True, muda_processo=True, reversivel=True,
      nota='Fila de pedidos parados em Aguardando. Pedido já existe, cliente já decidiu comprar.'),

 dict(id='I03', titulo='Recuperação de pedidos cancelados no pagamento', hip='H9',
      chave_rs='h9_cancelado_margem_ano', natureza='teto', capacidade='recuperacao_pos_venda',
      mecanismo='fila_operacional', bases=['vendas'],
      dep_terceiro=True, dado_novo=False, requer_adocao=True, muda_processo=True, reversivel=True,
      nota='Teto: parte do cancelamento é irrecuperável (limite de crédito, desistência). Depende do meio de pagamento.'),

 dict(id='I04', titulo='Redução da devolução por causa operacional', hip='H9',
      chave_rs='h9_dev_enderecavel_ano', natureza='teto', capacidade='prevencao_de_devolucao',
      mecanismo='processo_fornecedor', bases=['vendas'],
      dep_terceiro=True, dado_novo=False, requer_adocao=True, muda_processo=True, reversivel=False,
      nota='Defeito + tamanho errado + atraso. Teto: não se elimina 100% da devolução. Arrependimento e "não gostei" ficam fora.'),

 dict(id='I05', titulo='Teto de desconto por faixa de ticket', hip='H2',
      chave_rs='desconto_excedente_teto20_ano', natureza='valor recuperável', capacidade='regra_comercial',
      mecanismo='regra_politica', bases=['vendas'],
      dep_terceiro=False, dado_novo=False, requer_adocao=True, muda_processo=True, reversivel=True,
      nota='Excedente acima de 20%, só em pedidos que viraram caixa (sem dupla contagem com a H9). Recuperável 1:1 pela identidade contábil.'),

 dict(id='I06', titulo='Reposição dirigida aos SKUs de curva A em ruptura', hip='H7',
      chave_rs='ruptura_em_ruptura_margem_realizada_ano', natureza='perda medida', capacidade='alerta_de_estoque',
      mecanismo='alerta_operacional', bases=['vendas', 'estoque'],
      dep_terceiro=True, dado_novo=True, requer_adocao=False, muda_processo=True, reversivel=True,
      nota='Só os SKUs com status Ruptura (perda corrente). Os de Estoque Crítico são exposição, tratados na sensibilidade.'),

 dict(id='I07', titulo='Renegociação do frete do canal Marketplace', hip='H4',
      chave_rs='mkt_gap_rs_ano', natureza='valor recuperável', capacidade='negociacao_logistica',
      mecanismo='negociacao_comercial', bases=['vendas'],
      dep_terceiro=True, dado_novo=False, requer_adocao=False, muda_processo=False, reversivel=True,
      nota='Achado colateral medido no segmento de Marketing. Não responde à pergunta da H4 — é ação comercial, não de dados.'),

 dict(id='I08', titulo='Atacar a origem do chamado por falha operacional', hip='H10',
      chave_rs='atend_custo_falha_ano', natureza='custo recorrente', capacidade='prevencao_de_devolucao',
      mecanismo='processo_fornecedor', bases=['atendimento', 'vendas'],
      dep_terceiro=True, dado_novo=False, requer_adocao=True, muda_processo=True, reversivel=False,
      nota='Mesma cadeia causal da devolução por atraso/defeito: o chamado é sintoma do mesmo evento.'),

 dict(id='I09', titulo='Migração do volume simples para o ChatBot já em produção', hip='H11',
      chave_rs='chatbot_economia_ano', natureza='economia', capacidade='atendimento_automatizado',
      mecanismo='roteamento_existente', bases=['atendimento'],
      dep_terceiro=False, dado_novo=False, requer_adocao=False, muda_processo=True, reversivel=True,
      nota='Roteamento, não tecnologia nova: o bot já opera com CSAT >= humano a uma fração do custo.'),

 dict(id='I10', titulo='Triagem de chamados por regra determinística', hip='H14',
      chave_rs=None, natureza='habilitador', capacidade='atendimento_automatizado',
      mecanismo='regra_triagem', bases=['atendimento'],
      dep_terceiro=False, dado_novo=False, requer_adocao=False, muda_processo=True, reversivel=True,
      nota='Por regra, não por classificador: o texto da base é template repetido, não linguagem livre.'),

 dict(id='I11', titulo='Instrumentação da atribuição de marketing', hip='H5',
      chave_rs=None, natureza='habilitador', capacidade='contrato_de_dado',
      mecanismo='instrumentacao_dado', bases=['marketing', 'vendas'],
      dep_terceiro=True, dado_novo=True, requer_adocao=True, muda_processo=True, reversivel=True,
      nota='Registrar campanha e conversão no pedido. Pré-requisito de qualquer leitura de funil, CAC ou ROAS.'),

 dict(id='I12', titulo='Retenção dirigida por segmento de valor', hip='H6',
      chave_rs=None, natureza='exposição', capacidade='retencao_por_segmento',
      mecanismo='campanha_segmentada', bases=['clientes'],
      dep_terceiro=False, dado_novo=True, requer_adocao=True, muda_processo=True, reversivel=True,
      nota='LTV exposto é exposição de cadastro, não perda medida — e Vendas não tem granularidade de cliente para validar.'),

 dict(id='I13', titulo='Rotina semanal de decisão sobre os indicadores', hip='H13',
      chave_rs=None, natureza='habilitador', capacidade='regua_de_margem',
      mecanismo='rotina_decisao', bases=['vendas'],
      dep_terceiro=False, dado_novo=False, requer_adocao=True, muda_processo=True, reversivel=True,
      nota='A hipótese confirmada é que os sinais já existem e são ignorados. O que falta é o ritual, não o dado.'),
]

df = pd.DataFrame(CATALOGO)
df['veredito'] = df['hip'].map(lambda h: HIP[h]['veredito'])
df['regra_aplicada'] = df['veredito'].map(REGRAS)
# lê do CATALOGO (não da coluna) porque o pandas converte None em NaN, que é truthy
df['rs_ano'] = [float(K(c['chave_rs'])) if c['chave_rs'] else 0.0 for c in CATALOGO]
df['chave_rs'] = [c['chave_rs'] or '-' for c in CATALOGO]

print('{} iniciativas derivadas de {} hipóteses'.format(len(df), len(HIP)))
print()
print(df[['id', 'hip', 'veredito', 'titulo', 'chave_rs', 'rs_ano', 'natureza']].to_string(index=False))

13 iniciativas derivadas de 14 hipóteses

 id hip     veredito                                                   titulo                                chave_rs    rs_ano          natureza
I01  H9   Confirmada                  Redefinir a métrica de margem realizada                                       -      0.00       habilitador
I02  H9   Confirmada            Recuperação de pedidos com pagamento pendente                h9_aguardando_margem_ano 320759.97      perda medida
I03  H9   Confirmada           Recuperação de pedidos cancelados no pagamento                 h9_cancelado_margem_ano 653067.07              teto
I04  H9   Confirmada               Redução da devolução por causa operacional                  h9_dev_enderecavel_ano 989664.15              teto
I05  H2   Confirmada                     Teto de desconto por faixa de ticket           desconto_excedente_teto20_ano 291186.59 valor recuperável
I06  H7      Parcial        Reposição dirigida aos SKUs de curva A em ruptura rupt

### 1.2 O que foi deliberadamente descartado

Toda hipótese refutada gera uma **decisão registrada**, não uma iniciativa. Registrar isto
explicitamente tem valor consultivo: dizer à diretoria onde *não* investir é tão útil quanto
dizer onde investir, e evita que a frente volte à mesa daqui a três meses.

In [4]:
com_iniciativa = set(df['hip'])
descarte = []
for hid, h in HIP.items():
    if h['veredito'] == 'Refutada':
        motivo = next((s['motivo'] for s in IMPACTO['sem_rs'] if s['hipotese_id'] == hid), '')
        descarte.append(dict(hipotese=hid, segmento=h['segmento'], hipotese_txt=h['hipotese'],
                             decisao='Não investir em frente de escopo amplo', evidencia=motivo))
    elif hid not in com_iniciativa:
        descarte.append(dict(hipotese=hid, segmento=h['segmento'], hipotese_txt=h['hipotese'],
                             decisao='Coberta por iniciativa de outra hipótese do mesmo segmento', evidencia=''))

dfd = pd.DataFrame(descarte)
print('{} hipóteses sem iniciativa própria:'.format(len(dfd)))
print()
for _, r in dfd.iterrows():
    print('  [{}] {}'.format(r['hipotese'], r['hipotese_txt']))
    print('       decisão: {}'.format(r['decisao']))
    if r['evidencia']:
        print('       evidência: {}'.format(r['evidencia']))

cobertas = set(dfd['hipotese']) | com_iniciativa
assert cobertas == set(HIP), 'Hipótese sem destino: ' + str(set(HIP) - cobertas)
print()
print('OK: as {} hipóteses estão todas endereçadas (iniciativa ou decisão registrada).'.format(len(HIP)))

4 hipóteses sem iniciativa própria:

  [H1] O mix de produtos migrou para categorias menos rentáveis
       decisão: Não investir em frente de escopo amplo
       evidência: 0,34 p.p. entre a melhor e a pior categoria — não é driver.
  [H3] O custo do fornecedor subiu e não foi repassado ao preço
       decisão: Não investir em frente de escopo amplo
       evidência: CMV estável em 43,8% da receita; custo cadastrado não bate com o praticado (corr 0,005).
  [H8] Prazo de fornecedor e entrega lenta geram indisponibilidade
       decisão: Não investir em frente de escopo amplo
       evidência: entrega lenta não gera mais devolução (1,3 p.p.); efeito de fornecedor lento é pequeno (V=0,049).
  [H12] O atendimento ruim antecede a perda do cliente
       decisão: Não investir em frente de escopo amplo
       evidência: sem associação estatística (qui-quadrado p=0,11); churn sub-representado no grupo de má experiência.

OK: as 14 hipóteses estão todas endereçadas (iniciativa ou decisão regis

## 2. Sobreposição — medida, não declarada

O diagnóstico dizia "não somar: margem não realizada e desconto se sobrepõem". Isso é uma
ressalva defensiva: protege contra o erro, mas impede consolidar qualquer total.

Aqui a sobreposição vira número. Duas foram medidas diretamente na base (seção 14 do `06`) e
uma é estrutural, resolvida por construção do catálogo:

1. **Desconto × margem não realizada:** parte do desconto foi dada em pedidos que nunca
   viraram caixa. Medido: essa fatia já está dentro da H9. Por isso a I05 usa a chave que
   considera **apenas pedidos realizados** — a dupla contagem é eliminada na origem.
2. **Decomposição da H9:** devolução, cancelado e aguardando são partições disjuntas do
   mesmo total — somam exatamente a H9, sem interseção.
3. **Devolução por atraso × custo de chamado por falha (I04 × I08):** mesma cadeia causal,
   medidas em linhas diferentes do resultado (margem perdida vs. custo incorrido). Não é
   dupla contagem contábil, mas as duas caem juntas se a causa for atacada — por isso são
   agrupadas na mesma capacidade, e o ganho conjunto nunca é contado duas vezes.

In [5]:
h9_total = K('margem_gap_rs_ano')
partes = [K('h9_devolucao_margem_ano'), K('h9_cancelado_margem_ano'), K('h9_aguardando_margem_ano')]
print('Partição da H9 (R$/ano):')
print('  devolução           {:>12,.0f}'.format(partes[0]))
print('  pagamento cancelado {:>12,.0f}'.format(partes[1]))
print('  pagamento aguardando{:>12,.0f}'.format(partes[2]))
print('  soma                {:>12,.0f}   vs   H9 canônica {:>12,.0f}'.format(sum(partes), h9_total))
assert abs(sum(partes) - h9_total) < 1.0, 'partição da H9 não fecha'
print('  -> partição fecha: são disjuntas, somam o total, sem interseção.')
print()
dev_end, dev_nao = K('h9_dev_enderecavel_ano'), K('h9_dev_nao_enderecavel_ano')
print('Dentro da devolução: endereçável {:,.0f} + não endereçável {:,.0f} = {:,.0f}'.format(dev_end, dev_nao, dev_end + dev_nao))
assert abs((dev_end + dev_nao) - partes[0]) < 1.0
print()
print('Desconto — sobreposição MEDIDA com a H9:')
print('  desconto total/ano              {:>12,.0f}'.format(K('desconto_total_rs_ano')))
print('  ...dentro de pedidos não-caixa  {:>12,.0f}  ({}% do desconto) <- já contado na H9'.format(
      K('desconto_sobreposto_nao_realizado_ano'), K('desconto_sobreposto_pct')))
print('  ...em pedidos realizados        {:>12,.0f}  <- base da I05, sem dupla contagem'.format(K('desconto_adicional_realizado_ano')))
print()
print('  I05 usa o EXCEDENTE acima do teto, não o desconto inteiro: R$ {:,.0f}/ano'.format(K('desconto_excedente_teto20_ano')))

Partição da H9 (R$/ano):
  devolução              1,425,548
  pagamento cancelado      653,067
  pagamento aguardando     320,760
  soma                   2,399,375   vs   H9 canônica    2,399,375
  -> partição fecha: são disjuntas, somam o total, sem interseção.

Dentro da devolução: endereçável 989,664 + não endereçável 435,884 = 1,425,548

Desconto — sobreposição MEDIDA com a H9:
  desconto total/ano                 1,531,877
  ...dentro de pedidos não-caixa       379,560  (24.8% do desconto) <- já contado na H9
  ...em pedidos realizados           1,152,316  <- base da I05, sem dupla contagem

  I05 usa o EXCEDENTE acima do teto, não o desconto inteiro: R$ 291,187/ano


### 2.1 Sobreposição I08 × I09 — o resíduo, não o bruto

I08 ("atacar a origem do chamado por falha operacional") e I09 ("migrar volume simples para o ChatBot") não são independentes: uma fração dos tickets que I08 eliminaria na origem é a mesma fração de volume que I09 já conta como economia por migração de canal. Contar os dois pelo valor bruto duplica parte do mesmo ganho.

In [5]:
i08_bruto = K('atend_custo_falha_ano')
i09 = K('chatbot_economia_ano')
i08_residual = round(i08_bruto - i09, 2)

print('I08 ataca a origem do chamado de falha operacional (categoria_problema em falha).')
print('I09 migra para o ChatBot os tickets de "Onde está meu pedido?" hoje fora do ChatBot.')
print('Uma parte dos tickets que I08 elimina na origem é a MESMA fatia de volume que I09')
print('já conta como economia por migração de canal -- resolver a causa apaga o volume que')
print('o bot deixaria de atender.')
print()
print('  I08 bruto  (custo total da falha operacional, R$/ano)    {:>12,.2f}'.format(i08_bruto))
print('  I09        (economia de migração para o ChatBot, R$/ano) {:>12,.2f}'.format(i09))
print('  I08 residual = I08 bruto - I09 (R$/ano)                  {:>12,.2f}'.format(i08_residual))
print()
print('  -> I08 entra na régua pelo valor RESIDUAL: o que sobra depois de descontar a')
print('     fatia que I09 já captura. Evita contar a mesma redução de tickets duas vezes.')

I08 ataca a origem do chamado de falha operacional (categoria_problema em falha).
I09 migra para o ChatBot os tickets de "Onde está meu pedido?" hoje fora do ChatBot.
Uma parte dos tickets que I08 elimina na origem é a MESMA fatia de volume que I09
já conta como economia por migração de canal -- resolver a causa apaga o volume que
o bot deixaria de atender.

  I08 bruto  (custo total da falha operacional, R$/ano)      106,702.67
  I09        (economia de migração para o ChatBot, R$/ano)    46,043.33
  I08 residual = I08 bruto - I09 (R$/ano)                     60,659.34

  -> I08 entra na régua pelo valor RESIDUAL: o que sobra depois de descontar a
     fatia que I09 já captura. Evita contar a mesma redução de tickets duas vezes.

## 3. Os quatro critérios

Quatro perguntas que uma diretoria faz sobre qualquer iniciativa, e como cada uma é medida:

| Critério | Pergunta | Como é medido | Origem |
|---|---|---|---|
| **Impacto** | quanto vale? | R$/ano de EBITDA, em faixas absolutas | derivado da base |
| **Esforço** | o que custa fazer? | pessoa-dias do arquétipo de execução | regra declarada |
| **Velocidade** | em quanto tempo entra o primeiro real? | dias até o primeiro resultado | regra declarada |
| **Risco** | qual a chance de não se confirmar? | atributos observáveis + natureza do número | regra declarada |

**Faixas absolutas, não relativas à amostra.** Cada nota vem de um corte fixo. Se uma
iniciativa entrar ou sair da lista, as notas das outras não se mexem — o oposto de uma
normalização min-max, em que incluir um item lento rebaixa todo mundo.

In [6]:
# --- IMPACTO: materialidade contra a própria margem da empresa, não escala arbitrária ---
# A régua é a margem de contribuição realizada anual. Os cortes são os que um CFO usa para
# decidir se um número "mexe no ano": 10% da margem é material, 0,5% é ruído.
CORTES_MATERIALIDADE = [(10.0, 10), (5.0, 8), (2.0, 6), (1.0, 4), (0.5, 2), (0.0001, 1)]
CORTES_ESFORCO = [(10, 10), (20, 8), (30, 6), (45, 4), (60, 2)]     # <= limite -> nota
CORTES_VELOC   = [(30, 10), (45, 8), (60, 6), (90, 4), (120, 2)]    # <= limite -> nota

def banda(valor, cortes):
    """cortes: lista (limite, nota) em ordem decrescente. Retorna a 1ª nota cujo limite é atingido."""
    for limite, nota in cortes:
        if valor >= limite:
            return nota
    return 0

def banda_inversa(valor, cortes):
    for limite, nota in cortes:
        if valor <= limite:
            return nota
    return 1

# --- risco: 10 = risco mínimo. Penalidades por atributo observável e por natureza do número.
PENALIDADE = {'dep_terceiro': 2, 'dado_novo': 3, 'requer_adocao': 1, 'muda_processo': 1}
PENALIDADE_NATUREZA = {'teto': 2, 'exposição': 3, 'habilitador': 0,
                       'perda medida': 0, 'valor recuperável': 0, 'custo recorrente': 0, 'economia': 0}
BONUS_REVERSIVEL = 1

def nota_risco(r):
    n = 10.0
    for atributo, peso in PENALIDADE.items():
        if r[atributo]:
            n -= peso
    n -= PENALIDADE_NATUREZA.get(r['natureza'], 0)
    if r['reversivel']:
        n += BONUS_REVERSIVEL
    return float(np.clip(n, 0, 10))

df['pessoa_dias']   = df['mecanismo'].map(lambda m: MECANISMOS[m][0])
df['dias_primeiro'] = df['mecanismo'].map(lambda m: MECANISMOS[m][1])
df['pct_regua']       = (100 * df['rs_ano'] / REGUA).round(2)
df['nota_impacto']    = df['pct_regua'].map(lambda x: banda(x, CORTES_MATERIALIDADE))
df['nota_esforco']    = df['pessoa_dias'].map(lambda x: banda_inversa(x, CORTES_ESFORCO))
df['nota_velocidade'] = df['dias_primeiro'].map(lambda x: banda_inversa(x, CORTES_VELOC))
df['nota_risco']      = df.apply(nota_risco, axis=1)

print('Régua: R$ {:,.0f}/ano  |  1 ponto de materialidade = R$ {:,.0f}'.format(REGUA, REGUA / 100))
print()
print(df[['id', 'titulo', 'rs_ano', 'pct_regua', 'nota_impacto', 'pessoa_dias', 'nota_esforco',
          'dias_primeiro', 'nota_velocidade', 'nota_risco']].to_string(index=False))

Régua: R$ 7,212,700/ano  |  1 ponto de materialidade = R$ 72,127

 id                                                   titulo    rs_ano  pct_regua  nota_impacto  pessoa_dias  nota_esforco  dias_primeiro  nota_velocidade  nota_risco
I01                  Redefinir a métrica de margem realizada      0.00       0.00             0           12             8             30               10         9.0
I02            Recuperação de pedidos com pagamento pendente 320759.97       4.45             6           20             8             30               10         9.0
I03           Recuperação de pedidos cancelados no pagamento 653067.07       9.05             8           20             8             30               10         5.0
I04               Redução da devolução por causa operacional 989664.15      13.72            10           40             4             90                4         4.0
I05                     Teto de desconto por faixa de ticket 291186.59       4.04             6    

### 3.1 Dois eixos, não um número só

Um erro comum — e que a primeira versão deste notebook cometia — é somar os quatro critérios
num score único. O problema é aritmético: **três dos quatro critérios medem facilidade** e
apenas um mede valor. Com pesos parecidos, "fácil" recebe 60% do peso e "quanto vale" recebe
40%, e uma iniciativa de R$ 46 mil passa à frente de uma de R$ 990 mil por ser rápida.

A correção é separar o que é de natureza diferente:

- **Eixo de impacto** = materialidade em R$/ano contra a margem realizada da empresa.
- **Eixo de exequibilidade** = esforço (0,40), velocidade (0,30) e risco (0,30) — os três
  descrevem a mesma pergunta ("qual a chance de isso chegar ao fim dentro do horizonte") e
  por isso pertencem ao mesmo eixo, não a três eixos independentes.

Assim o *trade-off* fica visível em vez de ser dissolvido numa média. Uma frente grande e
difícil não vira "média", vira **estrutural** — e é tratada como tal no sequenciamento.

O limiar entre alto e baixo é **5,5**, o ponto médio teórico da escala, não a mediana da
amostra: se uma iniciativa entrar ou sair da lista, as fronteiras não se movem.

In [7]:
PESOS_EXEQ = {'nota_esforco': 0.40, 'nota_velocidade': 0.30, 'nota_risco': 0.30}
assert abs(sum(PESOS_EXEQ.values()) - 1.0) < 1e-9
LIMIAR = 5.5

def exequibilidade(frame, pesos=PESOS_EXEQ):
    return sum(p * frame[c] for c, p in pesos.items()).round(2)

df['eixo_impacto'] = df['nota_impacto'].astype(float)
df['eixo_exeq'] = exequibilidade(df)

def classifica(r):
    if r['rs_ano'] <= 0:
        return 'Habilitador (sem R$ próprio)'
    alto_i = r['eixo_impacto'] >= LIMIAR
    alta_e = r['eixo_exeq'] >= LIMIAR
    if alto_i and alta_e:   return 'Ganho imediato'
    if alto_i and not alta_e: return 'Estrutural'
    if not alto_i and alta_e: return 'Ganho marginal'
    return 'Baixa prioridade'

df['classe'] = df.apply(classifica, axis=1)
df = df.sort_values(['eixo_impacto', 'eixo_exeq'], ascending=False).reset_index(drop=True)
df.insert(0, 'pos', range(1, len(df) + 1))

print(df[['pos', 'id', 'titulo', 'rs_ano', 'pct_regua', 'eixo_impacto', 'eixo_exeq', 'classe']].to_string(index=False))
print()
print('Distribuição:', df['classe'].value_counts().to_dict())

 pos  id                                                   titulo    rs_ano  pct_regua  eixo_impacto  eixo_exeq                       classe
   1 I04               Redução da devolução por causa operacional 989664.15      13.72          10.0        4.0                   Estrutural
   2 I03           Recuperação de pedidos cancelados no pagamento 653067.07       9.05           8.0        7.7               Ganho imediato
   3 I02            Recuperação de pedidos com pagamento pendente 320759.97       4.45           6.0        8.9               Ganho imediato
   4 I05                     Teto de desconto por faixa de ticket 291186.59       4.04           6.0        8.9               Ganho imediato
   5 I07               Renegociação do frete do canal Marketplace 139417.93       1.93           4.0        7.7               Ganho marginal
   6 I06        Reposição dirigida aos SKUs de curva A em ruptura 120022.30       1.66           4.0        6.3               Ganho marginal
   7 I08     

## 4. Quanto vale a agenda, sem dupla contagem

Três leituras diferentes, porque juntar tudo num número só é o erro que o diagnóstico
anterior cometia:

- **Capturável no horizonte** — só as iniciativas de ganho imediato, cujo mecanismo é regra
  ou rotina sobre dado que já existe.
- **Endereçável total** — soma de todas as frentes com R$ medido, incluindo as estruturais.
- **Fora do somável** — exposição e frentes cuja natureza é teto, mantidas à parte.

A partição da H9 (seção 2) garante que não há interseção entre as parcelas.

In [8]:
com_rs = df[df['rs_ano'] > 0]
imediato   = com_rs[com_rs['classe'] == 'Ganho imediato']
estrutural = com_rs[com_rs['classe'] == 'Estrutural']

tot_imediato = imediato['rs_ano'].sum()
tot_geral    = com_rs['rs_ano'].sum()

print('CAPTURÁVEL NO HORIZONTE (ganhos imediatos)')
for _, r in imediato.iterrows():
    print('  {:<52} R$ {:>11,.0f}/ano   {:>5.2f}% da margem'.format(r['titulo'][:52], r['rs_ano'], r['pct_regua']))
print('  {:<52} R$ {:>11,.0f}/ano   {:>5.2f}% da margem'.format('TOTAL', tot_imediato, 100 * tot_imediato / REGUA))
print()
print('ESTRUTURAL (alto valor, exequibilidade baixa)')
for _, r in estrutural.iterrows():
    print('  {:<52} R$ {:>11,.0f}/ano   {:>5.2f}% da margem'.format(r['titulo'][:52], r['rs_ano'], r['pct_regua']))
print()
print('ENDEREÇÁVEL TOTAL (todas as frentes com R$ medido)')
print('  R$ {:,.0f}/ano  =  {:.1f}% da margem de contribuição realizada'.format(tot_geral, 100 * tot_geral / REGUA))
print()
print('FORA DO SOMÁVEL (mantidos à parte, por natureza)')
print('  Exposição de LTV em segmentos de risco:  R$ {:,.0f} (cadastro de clientes, não perda medida)'.format(K('rfm_ltv_exposto')))
print('  Ruptura incluindo Estoque Crítico:       R$ {:,.0f}/ano (exposição condicional — ver sensibilidade)'.format(K('ruptura_curvaA_margem_realizada_ano')))
print()
print('Natureza de cada frente somada:')
for _, r in com_rs.iterrows():
    print('  {:<52} {}'.format(r['titulo'][:52], r['natureza']))

CAPTURÁVEL NO HORIZONTE (ganhos imediatos)
  Recuperação de pedidos cancelados no pagamento       R$     653,067/ano    9.05% da margem
  Recuperação de pedidos com pagamento pendente        R$     320,760/ano    4.45% da margem
  Teto de desconto por faixa de ticket                 R$     291,187/ano    4.04% da margem
  TOTAL                                                R$   1,265,014/ano   17.54% da margem

ESTRUTURAL (alto valor, exequibilidade baixa)
  Redução da devolução por causa operacional           R$     989,664/ano   13.72% da margem

ENDEREÇÁVEL TOTAL (todas as frentes com R$ medido)
  R$ 2,666,864/ano  =  37.0% da margem de contribuição realizada

FORA DO SOMÁVEL (mantidos à parte, por natureza)
  Exposição de LTV em segmentos de risco:  R$ 27,801,945 (cadastro de clientes, não perda medida)
  Ruptura incluindo Estoque Crítico:       R$ 867,128/ano (exposição condicional — ver sensibilidade)

Natureza de cada frente somada:
  Redução da devolução por causa operacional 

### 4.1 Recalibrando o endereçável e a cascata D/V/F com o residual de I08

Consequência direta da seção 2.1: o ENDEREÇÁVEL TOTAL da seção 4 usava o I08 bruto. Substituindo pelo residual, o total cai exatamente pelo valor da sobreposição (= I09), e a cascata D/V/F dos dois módulos que dependem desse número (M1 e M2, seção 6) muda em conjunto — o vencedor do ranking não muda.

In [9]:
tot_geral_corrigido = round(tot_geral - i09, 2)   # i09 é exatamente a sobreposição (seção 2.1)
print('ENDEREÇÁVEL TOTAL, sem a dupla contagem I08 x I09:')
print('  publicado (I08 bruto)     R$ {:>13,.2f}/ano  =  {:.1f}% da margem'.format(tot_geral, 100*tot_geral/REGUA))
print('  sobreposição (= I09)      R$ {:>13,.2f}/ano'.format(i09))
print('  corrigido (I08 residual)  R$ {:>13,.2f}/ano  =  {:.1f}% da margem'.format(tot_geral_corrigido, 100*tot_geral_corrigido/REGUA))
print()
print('Efeito em cascata sobre D/V/F dos dois módulos que usam I08 (M2) e do módulo que')
print('antes dividia o topo do ranking de valor bruto (M1, que não usa I08). Isolados aqui')
print('porque MODULOS só existe a partir da seção 5 -- a seção 6 recalcula os dois com a')
print('cascata completa e chega nos mesmos números:')
print()

i02, i03, i04 = K('h9_aguardando_margem_ano'), K('h9_cancelado_margem_ano'), K('h9_dev_enderecavel_ano')
m1_rs = i02 + i03                                                  # M1 = I02 + I03 (40 pessoa-dias)
m2_rs_bruto, m2_rs_residual = i04 + i08_bruto, i04 + i08_residual  # M2 = I04 + I08 (80 pessoa-dias)
m1_pdias, m2_pdias = 40, 80
F_M1, F_M2 = 7.50, 10.00   # calculados na seção 6 (nota_F); não dependem de rs_ano

def dvf_par(m2_rs):
    rs_max  = max(m1_rs, m2_rs)
    vpd_max = max(m1_rs / m1_pdias, m2_rs / m2_pdias)
    d1, v1 = 10 * m1_rs / rs_max, 10 * (m1_rs / m1_pdias) / vpd_max
    d2, v2 = 10 * m2_rs / rs_max, 10 * (m2_rs / m2_pdias) / vpd_max
    dvf1 = 3 / (1/d1 + 1/v1 + 1/F_M1)
    dvf2 = 3 / (1/d2 + 1/v2 + 1/F_M2)
    return (d1, v1, dvf1), (d2, v2, dvf2)

(d1b, v1b, dvf1b), (d2b, v2b, dvf2b) = dvf_par(m2_rs_bruto)
(d1c, v1c, dvf1c), (d2c, v2c, dvf2c) = dvf_par(m2_rs_residual)

print('{:<8} {:<10} {:>6} {:>6} {:>6} {:>6}'.format('MÓDULO', 'VERSÃO', 'D', 'V', 'F', 'DVF'))
print('{:<8} {:<10} {:>6.2f} {:>6.2f} {:>6.2f} {:>6.2f}'.format('M1', 'bruto', d1b, v1b, F_M1, dvf1b))
print('{:<8} {:<10} {:>6.2f} {:>6.2f} {:>6.2f} {:>6.2f}'.format('', 'residual', d1c, v1c, F_M1, dvf1c))
print('{:<8} {:<10} {:>6.2f} {:>6.2f} {:>6.2f} {:>6.2f}'.format('M2', 'bruto', d2b, v2b, F_M2, dvf2b))
print('{:<8} {:<10} {:>6.2f} {:>6.2f} {:>6.2f} {:>6.2f}'.format('', 'residual', d2c, v2c, F_M2, dvf2c))
print()
print('M1 = Recuperação de receita pós-venda   |   M2 = Prevenção de devolução na origem')
print()
print('M1 sobe de D porque M2 (o maior rs_ano do conjunto) encolhe; M2 perde V porque seu')
print('valor por pessoa-dia cai. O vencedor não muda: M1 segue à frente em ambas as versões.')

ENDEREÇÁVEL TOTAL, sem a dupla contagem I08 x I09:
  publicado (I08 bruto)     R$  2,666,864.01/ano  =  37.0% da margem
  sobreposição (= I09)      R$     46,043.33/ano
  corrigido (I08 residual)  R$  2,620,820.68/ano  =  36.3% da margem

Efeito em cascata sobre D/V/F dos dois módulos que usam I08 (M2) e do módulo que
antes dividia o topo do ranking de valor bruto (M1, que não usa I08). Isolados aqui
porque MODULOS só existe a partir da seção 5 -- a seção 6 recalcula os dois com a
cascata completa e chega nos mesmos números:

MÓDULO   VERSÃO          D      V      F    DVF
M1       bruto        8.88  10.00   7.50   8.67
         residual     9.27  10.00   7.50   8.79
M2       bruto       10.00   5.63  10.00   7.94
         residual    10.00   5.39  10.00   7.78

M1 = Recuperação de receita pós-venda   |   M2 = Prevenção de devolução na origem

M1 sobe de D porque M2 (o maior rs_ano do conjunto) encolhe; M2 perde V porque seu
valor por pessoa-dia cai. O vencedor não muda: M1 segue à fre

## 5. Módulos de solução

As iniciativas priorizadas são encaixadas em **módulos**. Um módulo não é um agrupamento temático: ele é
definido pela **decisão de negócio que passa a governar**. Isso importa porque o que se automatiza é uma
decisão — não uma iniciativa —, e é a decisão que determina com que frequência a solução é usada e qual
dado ela precisa ter embaixo.

Cada módulo declara: a decisão, a frequência com que ela é tomada hoje, as iniciativas que captura, as
bases de que depende, e o **degrau de automação** de que ela precisaria para ser bem resolvida.

In [9]:
# --- escada de automação: ordem, dado mínimo, exige histórico rotulado, descrição ---
# A escada mede AUTONOMIA da decisão — quanto o sistema resolve sozinho.
# Não confundir com a FORMA da solução (agente, serviço, relatório), tratada na seção 7:
# um agente de software pode operar em qualquer degrau desta escala.
NIVEIS = {
    'nada':          (0, 0, False, 'Status quo — a decisão continua como está'),
    'definicao':     (1, 4, False, 'Definir a métrica ou a política uma vez; não exige sistema'),
    'regra':         (2, 6, False, 'Política aplicada no fluxo do pedido'),
    'alerta':        (3, 7, False, 'O sistema avisa quando um limite é cruzado'),
    'recomendacao':  (4, 7, False, 'O sistema ordena a fila e recomenda a próxima ação, com decisão humana'),
    'predicao':      (5, 8, True,  'O sistema estima probabilidade — exige histórico rotulado'),
    'acao_autonoma': (6, 9, True,  'O sistema executa a ação sozinho dentro de um envelope'),
}
ORD  = {k: v[0] for k, v in NIVEIS.items()}
DMIN = {k: v[1] for k, v in NIVEIS.items()}
ROT  = {k: v[2] for k, v in NIVEIS.items()}
POR_ORDEM = sorted(NIVEIS, key=lambda k: ORD[k])

print('Escada de automação:')
for k in POR_ORDEM:
    o, d, r, desc = NIVEIS[k]
    print('  {}. {:12} dado>={:<2} {:11} {}'.format(o, k, d, 'com rótulo' if r else '', desc))

Escada de automação:
  0. nada         dado>=0              Status quo — a decisão continua como está
  1. definicao    dado>=4              Definir a métrica ou a política uma vez; não exige sistema
  2. regra        dado>=6              Política aplicada no fluxo do pedido
  3. alerta       dado>=7              O sistema avisa quando um limite é cruzado
  4. recomendacao dado>=7              O sistema ordena a fila e recomenda a próxima ação, com decisão humana
  5. predicao     dado>=8  com rótulo  O sistema estima probabilidade — exige histórico rotulado
  6. acao_autonoma dado>=9  com rótulo  O sistema executa a ação sozinho dentro de um envelope


In [10]:
# --- integridade de cada base, derivada das métricas canônicas (rubricas declaradas) ---
def integridade_marketing():
    d = K('mkt_receita_declarada_vs_real_x')
    for lim, nota in [(1.1, 10.0), (1.5, 7.0), (3.0, 4.0), (10.0, 2.0)]:
        if d <= lim:
            return nota
    return 0.0

def integridade_clientes():
    c = K('clientes_cobertura_pct')
    for lim, nota in [(80, 10.0), (50, 8.0), (20, 5.0), (5, 2.0)]:
        if c >= lim:
            return nota
    return 1.0

INTEGRIDADE = {
    'vendas':      10.0 if K('identidade_linhas_fora_1centavo') == 0 else 4.0,
    'atendimento':  9.0,   # volume e categoria completos; texto é template, não linguagem livre
    'estoque':      5.0,   # foto sem histórico de dias sem estoque -> teto declarado
    'clientes':    integridade_clientes(),
    'marketing':   integridade_marketing(),
}
ANCORA = {
    'vendas':      'as duas identidades contábeis fecham, {} linhas fora de um centavo'.format(K('identidade_linhas_fora_1centavo')),
    'atendimento': '{:,.0f} tickets categorizados, mas só {} frases distintas no texto'.format(K('atend_tickets_total'), K('atend_frases_distintas')),
    'estoque':     'é uma foto: sem histórico de dias sem estoque; cobertura mediana de {:,.0f} dias'.format(K('estoque_cobertura_mediana_dias')),
    'clientes':    'só {}% da base aparece em Vendas ({} identificadores para {:,.0f} pedidos)'.format(K('clientes_cobertura_pct'), K('clientes_ids_em_vendas'), K('pedidos_validos')),
    'marketing':   'declara {}x a receita real e {:,.0f}x as conversões reais'.format(K('mkt_receita_declarada_vs_real_x'), K('mkt_conversoes_vs_pedidos_x')),
}
for b in INTEGRIDADE:
    print('  {:12} {:>4.0f}   {}'.format(b, INTEGRIDADE[b], ANCORA[b]))

  vendas         10   as duas identidades contábeis fecham, 0 linhas fora de um centavo
  atendimento     9   35,840 tickets categorizados, mas só 30 frases distintas no texto
  estoque         5   é uma foto: sem histórico de dias sem estoque; cobertura mediana de 5,339 dias
  clientes        1   só 2.31% da base aparece em Vendas (346 identificadores para 27,758 pedidos)
  marketing       0   declara 42.8x a receita real e 4,153x as conversões reais


In [11]:
# Um módulo por decisão de negócio. 'precisa' = degrau necessário para resolver bem a decisão;
# 'rotulo' = existe histórico rotulado para treinar um modelo hoje.
MODULOS = [
 dict(id='M1', nome='Recuperação de receita pós-venda', inis=['I02', 'I03'],
      decisao='Este pedido parado ainda pode virar caixa?',
      freq=int(K('pedidos_validos') * K('pct_pedidos_nao_caixa') / 100),
      freq_origem='pedidos que não viraram caixa na janela', precisa='predicao', rotulo=False,
      nota='Priorizar a fila por valor e recência já é acionável hoje. Um modelo de propensão exige saber '
           'quais tentativas de recuperação deram certo — rótulo que só o piloto gera.'),

 dict(id='M2', nome='Prevenção de devolução na origem', inis=['I04', 'I08'],
      decisao='Por que este pedido voltou, e o que muda na origem?',
      freq=int(K('pedidos_validos') * K('devolucao_tx_global_pct') / 100),
      freq_origem='pedidos devolvidos na janela', precisa='alerta', rotulo=True,
      nota='Defeito, tamanho e atraso já vêm rotulados em motivo_devolucao — dá para monitorar por fornecedor e SKU.'),

 dict(id='M3', nome='Regra comercial de desconto', inis=['I05'],
      decisao='Que desconto conceder neste pedido?',
      freq=int(K('pedidos_validos') * K('desconto_pct_pedidos') / 100),
      freq_origem='pedidos com desconto na janela', precisa='regra', rotulo=False,
      nota='Teto por faixa aplicado no fluxo do pedido. Não precisa de modelo: o dado já diz que volume não responde a desconto.'),

 dict(id='M4', nome='Reposição de estoque em curva A', inis=['I06'],
      decisao='O que repor, e quando?', freq=52,
      freq_origem='ciclo semanal de reposição', precisa='alerta', rotulo=False,
      nota='O alerta depende de posição de estoque confiável ao longo do tempo, que a base não tem.'),

 dict(id='M5', nome='Atendimento automatizado', inis=['I09', 'I10'],
      decisao='Este chamado precisa de um atendente?', freq=int(K('atend_tickets_ano')),
      freq_origem='tickets por ano', precisa='regra', rotulo=False,
      nota='Roteamento e triagem por regra. Classificador de linguagem não se sustenta com texto template.'),

 dict(id='M6', nome='Régua de margem e rotina de decisão', inis=['I01', 'I13'],
      decisao='Qual é a margem que virou caixa, e o que a diretoria faz com ela?', freq=52,
      freq_origem='ciclo semanal de gestão', precisa='definicao', rotulo=False,
      nota='Corrige o número que a diretoria olha e institui o ritual. Não captura R$ próprio.'),

 dict(id='M7', nome='Renegociação de frete do canal', inis=['I07'],
      decisao='Aceitamos as condições de frete do canal?', freq=1,
      freq_origem='ciclo de contrato anual', precisa='definicao', rotulo=False,
      nota='Decisão anual: é negociação comercial, não software.'),

 dict(id='M8', nome='Alocação de verba de marketing', inis=['I11'],
      decisao='Onde alocar a verba de marketing?', freq=12,
      freq_origem='revisão mensal de mídia', precisa='predicao', rotulo=False,
      nota='Exige atribuição confiável entre campanha e pedido, que hoje não existe.'),

 dict(id='M9', nome='Retenção por segmento de valor', inis=['I12'],
      decisao='Quais clientes merecem esforço de retenção?', freq=12,
      freq_origem='ciclo mensal de CRM', precisa='predicao', rotulo=False,
      nota='Exige elo cliente-pedido em Vendas, que a base não sustenta.'),
]

POR_INI = {i['id']: i for i in df.to_dict('records')}
for m in MODULOS:
    subs = [POR_INI[i] for i in m['inis']]
    m['titulos']      = [s['titulo'] for s in subs]
    m['bases']        = sorted({b for s in subs for b in s['bases']})
    m['rs_ano']       = float(sum(s['rs_ano'] for s in subs))
    m['pessoa_dias']  = int(sum(s['pessoa_dias'] for s in subs))
    m['dias_primeiro'] = int(min(s['dias_primeiro'] for s in subs))
    m['dado']         = min(INTEGRIDADE[b] for b in m['bases'])
    m['dado_ancora']  = ANCORA[min(m['bases'], key=lambda b: INTEGRIDADE[b])]

print('{} módulos, cobrindo as {} iniciativas'.format(len(MODULOS), sum(len(m['inis']) for m in MODULOS)))

9 módulos, cobrindo as 13 iniciativas


### 5.1 Em que degrau cada módulo pode operar

Três limites, aplicados em ordem. O degrau em que o módulo opera é o menor entre eles — e é ele que dá o
**escopo concreto de construção**, em vez de um nome abstrato:

1. **O que a decisão precisa.** Não se constrói acima da necessidade.
2. **O que o dado sustenta.** Cada degrau exige uma integridade mínima; degraus de modelo exigem também
   histórico rotulado, que é coisa diferente de dado íntegro.
3. **O que a frequência justifica.** Decisão tomada uma vez por ano não se automatiza — vira definição.
   Abaixo de 100 vezes por ano, alerta é o teto razoável.

In [12]:
def teto_por_frequencia(freq):
    if freq < 12:  return ORD['definicao']
    if freq < 100: return ORD['alerta']
    return ORD['acao_autonoma']

def degrau_operado(m):
    limite = min(ORD[m['precisa']], teto_por_frequencia(m['freq']))
    for nivel in reversed(POR_ORDEM):
        if ORD[nivel] > limite:                      continue
        if DMIN[nivel] > m['dado']:                  continue
        if ROT[nivel] and not m['rotulo']:           continue
        return nivel
    return 'nada'

for m in MODULOS:
    m['degrau'] = degrau_operado(m)
    m['limitador'] = ('dado' if DMIN[m['precisa']] > m['dado']
                      else 'rótulo' if ROT[m['precisa']] and not m['rotulo']
                      else 'frequência' if teto_por_frequencia(m['freq']) < ORD[m['precisa']]
                      else 'nenhum — opera no degrau de que precisa')

print('{:<34} {:>9} {:>5}  {:<12} -> {:<12} {}'.format('MÓDULO', 'FREQ/ANO', 'DADO', 'PRECISA', 'OPERA', 'LIMITADOR'))
for m in MODULOS:
    print('{:<34} {:>9,} {:>5.0f}  {:<12} -> {:<12} {}'.format(
          m['nome'][:34], m['freq'], m['dado'], m['precisa'], m['degrau'], m['limitador']))

MÓDULO                              FREQ/ANO  DADO  PRECISA      -> OPERA        LIMITADOR
Recuperação de receita pós-venda       6,942    10  predicao     -> recomendacao rótulo
Prevenção de devolução na origem       4,127     9  alerta       -> alerta       nenhum — opera no degrau de que precisa
Regra comercial de desconto            9,851    10  regra        -> regra        nenhum — opera no degrau de que precisa
Reposição de estoque em curva A           52     5  alerta       -> definicao    dado
Atendimento automatizado              11,946     9  regra        -> regra        nenhum — opera no degrau de que precisa
Régua de margem e rotina de decisã        52    10  definicao    -> definicao    nenhum — opera no degrau de que precisa
Renegociação de frete do canal             1    10  definicao    -> definicao    nenhum — opera no degrau de que precisa
Alocação de verba de marketing            12     0  predicao     -> nada         dado
Retenção por segmento de valor            12

## 6. Avaliação DVF

Os três pilares, cada um de **uma fonte só** — sem sub-pesos e sem componentes compostos:

- **D · Desejabilidade** — R$/ano que o módulo captura, contra o maior da lista. É quanto da dor confirmada
  ele endereça.
- **V · Viabilidade** — R$/ano por pessoa-dia. É a pergunta "vale o trabalho", que é o que se responde
  honestamente sem dado de investimento.
- **F · Factibilidade** — distância entre o dado que a decisão tem e o que o degrau necessário exige.

Combinados por **média harmônica**: o pilar mais fraco domina e zera o resultado se for zero. É o que faz
um módulo apoiado em base quebrada morrer sem precisar de argumento — e é por isso que a factibilidade não
precisa de um mecanismo próprio de eliminação.

**Módulos sem R$ próprio não entram no ranking.** Eles não são piores: são de outra natureza — governança
ou pré-requisito. Dar-lhes um R$ inventado para que competissem seria o erro que este trabalho corrige.

In [13]:
def nota_F(m):
    falta = DMIN[m['precisa']] - m['dado']
    f = 10.0 - 2.5 * max(0.0, falta)
    if ROT[m['precisa']] and not m['rotulo']:
        f -= 2.5           # dado íntegro mas sem histórico rotulado: o modelo não treina hoje
    return float(np.clip(f, 0, 10))

competem = [m for m in MODULOS if m['rs_ano'] > 0]
apoio    = [m for m in MODULOS if m['rs_ano'] == 0]

rs_max  = max(m['rs_ano'] for m in competem)
vpd_max = max(m['rs_ano'] / m['pessoa_dias'] for m in competem)
for m in competem:
    m['D'] = round(10 * m['rs_ano'] / rs_max, 2)
    m['V'] = round(10 * (m['rs_ano'] / m['pessoa_dias']) / vpd_max, 2)
    m['F'] = round(nota_F(m), 2)
    m['DVF'] = 0.0 if min(m['D'], m['V'], m['F']) <= 0 else round(3 / (1/m['D'] + 1/m['V'] + 1/m['F']), 2)
    m['pilar_fraco'] = min(['D', 'V', 'F'], key=lambda p: m[p])
for m in apoio:
    m['D'] = m['V'] = m['F'] = m['DVF'] = 0.0
    m['pilar_fraco'] = '—'

competem.sort(key=lambda m: -m['DVF'])
for i, m in enumerate(competem, 1):
    m['rank'] = i
for m in apoio:
    m['rank'] = None

print('{:<34} {:>11} {:>5} {:>6} {:>5} {:>5} {:>6}  {}'.format('MÓDULO', 'R$/ANO', 'P-DIA', 'D', 'V', 'F', 'DVF', 'PILAR FRACO'))
for m in competem:
    print('{:<34} {:>11,.0f} {:>5} {:>6.2f} {:>5.2f} {:>5.2f} {:>6.2f}  {}'.format(
          m['nome'][:34], m['rs_ano'], m['pessoa_dias'], m['D'], m['V'], m['F'], m['DVF'], m['pilar_fraco']))
print()
print('Sem R$ próprio — fora do ranking de prioridade:')
for m in apoio:
    print('  {:<34} {}'.format(m['nome'][:34], m['nota']))

VENC = competem[0]
print()
print('RECOMENDAÇÃO: {} — DVF {:.2f}, operando no degrau "{}"'.format(VENC['nome'], VENC['DVF'], VENC['degrau']))

MÓDULO                                  R$/ANO P-DIA      D     V     F    DVF  PILAR FRACO
Recuperação de receita pós-venda       973,827    40   8.88 10.00  7.50   8.67  F
Prevenção de devolução na origem     1,096,367    80  10.00  5.63 10.00   7.94  V
Regra comercial de desconto            291,187    15   2.66  7.97 10.00   4.99  D
Renegociação de frete do canal         139,418    20   1.27  2.86 10.00   2.43  D
Reposição de estoque em curva A        120,022    25   1.09  1.97  5.00   1.85  D
Atendimento automatizado                46,043    25   0.42  0.76 10.00   0.79  D

Sem R$ próprio — fora do ranking de prioridade:
  Régua de margem e rotina de decisã Corrige o número que a diretoria olha e institui o ritual. Não captura R$ próprio.
  Alocação de verba de marketing     Exige atribuição confiável entre campanha e pedido, que hoje não existe.
  Retenção por segmento de valor     Exige elo cliente-pedido em Vendas, que a base não sustenta.

RECOMENDAÇÃO: Recuperação de receita p

### 6.1 Sensibilidade

Quatro cenários, os únicos que podem mudar a recomendação. Se o primeiro lugar não se mexe em nenhum
deles, a escolha não depende de um juízo isolado.

In [14]:
def refaz(fator_esforco=1.0, integridade_estoque=5.0,
          chave_ruptura='ruptura_em_ruptura_margem_realizada_ano',
          chave_desconto='desconto_excedente_teto20_ano'):
    integ = dict(INTEGRIDADE); integ['estoque'] = integridade_estoque
    rs_ini = {i: POR_INI[i]['rs_ano'] for i in POR_INI}
    rs_ini['I06'] = float(MET[chave_ruptura]['valor'])
    rs_ini['I05'] = float(MET[chave_desconto]['valor'])
    linhas = []
    for m in MODULOS:
        rs = sum(rs_ini[i] for i in m['inis'])
        if rs <= 0:
            continue
        pdias = sum(POR_INI[i]['pessoa_dias'] for i in m['inis']) * fator_esforco
        dado = min(integ[b] for b in m['bases'])
        f = 10.0 - 2.5 * max(0.0, DMIN[m['precisa']] - dado)
        if ROT[m['precisa']] and not m['rotulo']:
            f -= 2.5
        linhas.append(dict(nome=m['nome'], rs=rs, vpd=rs / pdias, F=float(np.clip(f, 0, 10))))
    rmax = max(x['rs'] for x in linhas); vmax = max(x['vpd'] for x in linhas)
    for x in linhas:
        D, V, F = 10 * x['rs'] / rmax, 10 * x['vpd'] / vmax, x['F']
        x['DVF'] = 0.0 if min(D, V, F) <= 0 else 3 / (1/D + 1/V + 1/F)
    return sorted(linhas, key=lambda x: -x['DVF'])

# Uma variável por cenário: cenário composto esconde qual driver causou a mudança.
CENARIOS = [
    ('esforço 40% maior em todos os módulos',       dict(fator_esforco=1.4)),
    ('esforço 40% menor em todos os módulos',       dict(fator_esforco=0.6)),
    ('teto de desconto em 15% em vez de 20%',       dict(chave_desconto='desconto_excedente_teto15_ano')),
    ('ruptura conta também o estoque crítico',      dict(chave_ruptura='ruptura_curvaA_margem_realizada_ano')),
    ('base de estoque tratada como confiável (8)',  dict(integridade_estoque=8.0)),
    ('as duas hipóteses otimistas de estoque juntas',
     dict(chave_ruptura='ruptura_curvaA_margem_realizada_ano', integridade_estoque=8.0)),
]
sens = []
for nome, kw in CENARIOS:
    top = refaz(**kw)[0]
    inverteu = top['nome'] != VENC['nome']
    sens.append(dict(cenario=nome, primeiro=top['nome'], dvf=round(top['DVF'], 2), inverteu=bool(inverteu)))
    print('  {:<44} -> {:<34} {}'.format(nome, top['nome'][:34], 'INVERTE' if inverteu else 'estável'))

N_INV = sum(s['inverteu'] for s in sens)
ESTAVEL = N_INV == 0
print()
print('{} de {} cenários invertem a recomendação.'.format(N_INV, len(sens)))

  esforço 40% maior em todos os módulos        -> Recuperação de receita pós-venda   estável
  esforço 40% menor em todos os módulos        -> Recuperação de receita pós-venda   estável
  teto de desconto em 15% em vez de 20%        -> Recuperação de receita pós-venda   estável
  ruptura conta também o estoque crítico       -> Recuperação de receita pós-venda   estável
  base de estoque tratada como confiável (8)   -> Recuperação de receita pós-venda   estável
  as duas hipóteses otimistas de estoque juntas -> Reposição de estoque em curva A    INVERTE

1 de 6 cenários invertem a recomendação.


### 6.2 A sensibilidade real: os 6 cenários com o residual de I08

Reexecuta os 6 cenários da seção 6.1 usando o `refaz()` já definido acima, sem reescrevê-lo — só sobrescrevendo temporariamente o `rs_ano` de I08 entre bruto e residual. Responde de forma definitiva se a correção da seção 2.1 muda a recomendação final ou a robustez já reportada ("1 de 6 cenários inverte").

In [14]:
def refaz_v4(i08_valor, **kw):
    """Mesma refaz() da seção 6.1, com o rs_ano de I08 sobrescrito (bruto ou residual)."""
    original = POR_INI['I08']['rs_ano']
    POR_INI['I08']['rs_ano'] = i08_valor
    try:
        return refaz(**kw)
    finally:
        POR_INI['I08']['rs_ano'] = original

print('Repetição dos 6 cenários da seção 6.1, agora também com o I08 residual (seção 2.1).')
print('A pergunta original -- "os 6 cenários envolvem I08 ou o módulo M2?" -- tinha resposta')
print('errada na primeira leitura: o M2 SEMPRE entra na conta de refaz(), só nunca vence.')
print()

top_base = refaz_v4(i08_bruto)[0]['nome']
print('{:<48} {:<10} {:<10}'.format('CENÁRIO', 'BRUTO', 'RESIDUAL'))
n_inv_bruto = n_inv_resid = 0
for nome, kw in CENARIOS:
    top_b = refaz_v4(i08_bruto, **kw)[0]
    top_r = refaz_v4(i08_residual, **kw)[0]
    inv_b, inv_r = top_b['nome'] != top_base, top_r['nome'] != top_base
    n_inv_bruto += inv_b; n_inv_resid += inv_r
    print('{:<48} {:<10} {:<10}'.format(nome, 'INVERTE' if inv_b else 'estável', 'INVERTE' if inv_r else 'estável'))

print()
print('{} de 6 cenários invertem com I08 bruto; {} de 6 invertem com I08 residual.'.format(n_inv_bruto, n_inv_resid))
print('Mesmo cenário inverte nos dois casos -- a correção do residual não muda quais cenários')
print('são frágeis, só refina o DVF de M2 (ver seção 4.1).')

Repetição dos 6 cenários da seção 6.1, agora também com o I08 residual (seção 2.1).
A pergunta original -- "os 6 cenários envolvem I08 ou o módulo M2?" -- tinha resposta
errada na primeira leitura: o M2 SEMPRE entra na conta de refaz(), só nunca vence.

CENÁRIO                                          BRUTO      RESIDUAL  
esforço 40% maior em todos os módulos            estável    estável   
esforço 40% menor em todos os módulos            estável    estável   
teto de desconto em 15% em vez de 20%            estável    estável   
ruptura conta também o estoque crítico           estável    estável   
base de estoque tratada como confiável (8)       estável    estável   
as duas hipóteses otimistas de estoque juntas    INVERTE    INVERTE   

1 de 6 cenários invertem com I08 bruto; 1 de 6 invertem com I08 residual.
Mesmo cenário inverte nos dois casos -- a correção do residual não muda quais cenários
são frágeis, só refina o DVF de M2 (ver seção 4.1).

## 7. Nível de complexidade e padrão da solução

O degrau diz **quanta autonomia** a decisão comporta. Falta classificar **que tipo de solução** isso
exige — porque é o tipo, e não a ambição, que determina custo, prazo e risco de adoção.

Três níveis, em ordem crescente de custo e prazo:

| Nível | O que é | Padrões típicos |
|---|---|---|
| **Interação** | prompt estruturado sobre um modelo de linguagem | LLM + recuperação de documentos · LLM + template de prompt · LLM + prompt de classificação |
| **Orquestração** | agente com ferramentas determinísticas | agente + acesso a ferramentas · agente + scoring estruturado · agente + pipeline de dados |
| **Engenharia** | solução dedicada, com infraestrutura própria | memória persistente · automação de interface · visão computacional |

O nível não é escolhido: **decorre do degrau**. Uma decisão que se resolve com política escrita não
precisa de solução de IA nenhuma — e dizer isso vale mais do que empurrar um modelo para dentro dela.

In [15]:
# Mapeia degrau de autonomia -> nível de complexidade + padrão de solução.
# Regra declarada: o degrau já expressa o que o sistema precisa fazer; o nível é consequência.
NIVEL_PADRAO = {
    'nada':          ('—',             'Sem solução — decisão bloqueada por integridade de dado'),
    'definicao':     ('—',             'Sem solução de IA — definição de métrica ou política'),
    'regra':         ('—',             'Sem solução de IA — regra determinística no fluxo'),
    'alerta':        ('Orquestração',  'Agente + pipeline de dados'),
    'recomendacao':  ('Orquestração',  'Agente + scoring estruturado'),
    'predicao':      ('Engenharia',    'Memória persistente'),
    'acao_autonoma': ('Engenharia',    'Memória persistente'),
}

for m in MODULOS:
    m['nivel'], m['padrao'] = NIVEL_PADRAO[m['degrau']]

print('{:<34} {:<14} {:<15} {}'.format('MÓDULO', 'DEGRAU', 'NÍVEL', 'PADRÃO'))
for m in competem + apoio:
    print('{:<34} {:<14} {:<15} {}'.format(m['nome'][:34], m['degrau'], m['nivel'], m['padrao']))

V = competem[0]
print()
print('Módulo recomendado: {} — nível {}, padrão "{}".'.format(V['nome'], V['nivel'], V['padrao']))
n_sem_ia = sum(1 for m in MODULOS if m['nivel'] == '—')
print('{} dos {} módulos não exigem solução de IA: são política, processo ou estão bloqueados por dado.'.format(
      n_sem_ia, len(MODULOS)))

MÓDULO                             DEGRAU         NÍVEL           PADRÃO
Recuperação de receita pós-venda   recomendacao   Orquestração    Agente + scoring estruturado
Prevenção de devolução na origem   alerta         Orquestração    Agente + pipeline de dados
Regra comercial de desconto        regra          —               Sem solução de IA — regra determinística no fluxo
Renegociação de frete do canal     definicao      —               Sem solução de IA — definição de métrica ou política
Reposição de estoque em curva A    definicao      —               Sem solução de IA — definição de métrica ou política
Atendimento automatizado           regra          —               Sem solução de IA — regra determinística no fluxo
Régua de margem e rotina de decisã definicao      —               Sem solução de IA — definição de métrica ou política
Alocação de verba de marketing     nada           —               Sem solução — decisão bloqueada por integridade de dado
Retenção por segmento de val

## 8. Sequência recomendada

O que paga rápido financia e dá credibilidade ao que demora. A ordem cruza a avaliação DVF com o tempo até
o primeiro resultado.

In [16]:
roadmap = []
for m in sorted(competem, key=lambda x: x['rank']) + apoio:
    if m['rs_ano'] == 0:
        janela = 'Acompanha a execução (não é ponto de partida)'
    elif m['DVF'] <= 0:
        janela = 'Fora do horizonte — bloqueado por dado'
    elif m['dias_primeiro'] <= 30:
        janela = '0-30 dias'
    elif m['dias_primeiro'] <= 60:
        janela = '31-60 dias'
    else:
        janela = '61-90 dias'
    roadmap.append(dict(janela=janela, modulo=m['nome'], degrau=m['degrau'], dvf=m['DVF'],
                        rs_ano=m['rs_ano'], pessoa_dias=m['pessoa_dias'], inis=m['inis']))

for j in ['0-30 dias', '31-60 dias', '61-90 dias', 'Acompanha a execução (não é ponto de partida)',
          'Fora do horizonte — bloqueado por dado']:
    itens = [r for r in roadmap if r['janela'] == j]
    if not itens:
        continue
    print(j.upper())
    for r in itens:
        print('   {:<34} {:<12} DVF {:>5.2f}   R$ {:>10,.0f}/ano   {:>3} p-dias'.format(
              r['modulo'][:34], r['degrau'], r['dvf'], r['rs_ano'], r['pessoa_dias']))
    print()

0-30 DIAS
   Recuperação de receita pós-venda   recomendacao DVF  8.67   R$    973,827/ano    40 p-dias
   Regra comercial de desconto        regra        DVF  4.99   R$    291,187/ano    15 p-dias
   Atendimento automatizado           regra        DVF  0.79   R$     46,043/ano    25 p-dias

31-60 DIAS
   Renegociação de frete do canal     definicao    DVF  2.43   R$    139,418/ano    20 p-dias
   Reposição de estoque em curva A    definicao    DVF  1.85   R$    120,022/ano    25 p-dias

61-90 DIAS
   Prevenção de devolução na origem   alerta       DVF  7.94   R$  1,096,367/ano    80 p-dias

ACOMPANHA A EXECUÇÃO (NÃO É PONTO DE PARTIDA)
   Régua de margem e rotina de decisã definicao    DVF  0.00   R$          0/ano    42 p-dias
   Alocação de verba de marketing     nada         DVF  0.00   R$          0/ano    35 p-dias
   Retenção por segmento de valor     nada         DVF  0.00   R$          0/ano    25 p-dias



## 9. Exportação

Grava `outputs/priorizacao.json`, fonte única da dashboard. Nenhum número da página é digitado à mão, e
todo valor aqui rastreia até `numeros_canonicos.json` pela chave registrada em `chaves_canonicas_usadas`.

In [17]:
EVIDENCIA_NARRATIVA = [
    'unidades_amplitude_faixas', 'devolucao_tx_global_pct', 'devolucao_tx_amplitude_canal_pp',
    'desconto_total_rs_ano', 'ruptura_curvaA_receita_ano', 'ruptura_em_ruptura_skus',
    'ruptura_estoque_critico_skus', 'margem_realizada_pct', 'margem_contabil_pct',
    'h9_dev_enderecavel_ano', 'h9_dev_nao_enderecavel_ano', 'chatbot_csat', 'humano_csat',
    'ruptura_curvaA_qtd', 'h9_devolucao_margem_ano', 'h9_cancelado_margem_ano',
    'h9_aguardando_margem_ano', 'desconto_excedente_teto20_ano', 'pct_pedidos_nao_caixa',
]
for chave in EVIDENCIA_NARRATIVA:
    K(chave)

limpa = lambda m: {k: v for k, v in m.items() if k != 'titulos'} | {'titulos': m['titulos']}
payload = {
    '_meta': {
        'gerado_por': '08_priorizacao_e_selecao.ipynb',
        'fonte': ['numeros_canonicos.json', 'impacto.json'],
        'principio': 'conclusão é saída do cálculo; nenhum quadrante, degrau ou vencedor é entrada',
        'sem_payback': 'investimento não existe em nenhuma base; payback é entregável do roadmap',
        'n_iniciativas': int(len(df)), 'n_modulos': len(MODULOS),
    },
    'regua': {'denominador_rs_ano': REGUA, 'label': 'margem de contribuição realizada anual'},
    'parametros': {
        'cortes_materialidade_pct': CORTES_MATERIALIDADE, 'cortes_esforco_pessoa_dias': CORTES_ESFORCO,
        'cortes_velocidade_dias': CORTES_VELOC, 'pesos_exequibilidade': PESOS_EXEQ,
        'limiar_quadrante': LIMIAR, 'penalidade_risco': PENALIDADE,
        'penalidade_natureza': PENALIDADE_NATUREZA, 'bonus_reversivel': BONUS_REVERSIVEL,
        'mecanismos_pessoa_dias': {k: v[0] for k, v in MECANISMOS.items()},
        'niveis': {k: {'ordem': v[0], 'dado_min': v[1], 'precisa_rotulo': v[2], 'desc': v[3]}
                   for k, v in NIVEIS.items()},
        'integridade_bases': INTEGRIDADE, 'ancora_bases': ANCORA,
    },
    'regras_derivacao': REGRAS,
    'iniciativas': json.loads(df.drop(columns=['regra_aplicada']).to_json(orient='records')),
    'descarte': json.loads(dfd.to_json(orient='records')),
    'modulos': [limpa(m) for m in competem + apoio],
    'nivel_padrao': NIVEL_PADRAO,
    'totais': {
        'capturavel_horizonte_rs_ano': float(tot_imediato),
        'capturavel_pct_margem': round(100 * tot_imediato / REGUA, 2),
        'enderecavel_total_rs_ano': float(tot_geral),
        'enderecavel_pct_margem': round(100 * tot_geral / REGUA, 2),
        'fora_do_somavel': {'ltv_exposto_rs': K('rfm_ltv_exposto'),
                            'ruptura_com_critico_rs_ano': K('ruptura_curvaA_margem_realizada_ano')},
    },
    'vencedor': {'modulo': VENC['nome'], 'dvf': VENC['DVF'], 'degrau': VENC['degrau'],
                 'decisao': VENC['decisao'], 'rs_ano': VENC['rs_ano'],
                 'pessoa_dias': VENC['pessoa_dias'], 'dias_primeiro': VENC['dias_primeiro'],
                 'inis': VENC['inis'], 'estavel_na_sensibilidade': bool(ESTAVEL),
                 'nivel': competem[0]['nivel'], 'padrao': competem[0]['padrao']},
    'sensibilidade': sens,
    'roadmap': roadmap,
    'series': {k: CANON['series'][k] for k in
               ['h9_decomposicao', 'devolucao_motivos', 'ruptura_leituras', 'desconto_tetos']},
    'chaves_canonicas_usadas': USADAS,
}
Path('outputs/priorizacao.json').write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('outputs/priorizacao.json gravado')
print('  {} iniciativas | {} módulos ({} competem) | {} chaves rastreadas'.format(
      len(df), len(MODULOS), len(competem), len(USADAS)))
print('  vencedor: {} (DVF {}) no degrau "{}" | estável: {}'.format(
      VENC['nome'], VENC['DVF'], VENC['degrau'], ESTAVEL))

outputs/priorizacao.json gravado
  13 iniciativas | 9 módulos (6 competem) | 41 chaves rastreadas
  vencedor: Recuperação de receita pós-venda (DVF 8.67) no degrau "recomendacao" | estável: False
